# AC-MOT v10 — five missing sequences + 17-sequence merge

Run all cells on a Tesla T4. This notebook runs only the five sequences missing from the preserved 12-sequence per-sequence results, then merges them without overwriting the old inputs.

The old 12-sequence CSVs must be uploaded somewhere under `/content/drive/MyDrive` before Cell 5.


In [ ]:
# ════════════════════════════════════════════════════════════════
#  CELL 1 — SETUP + FIVE MISSING SEQUENCES
# ════════════════════════════════════════════════════════════════
!pip install ultralytics motmetrics opencv-python-headless pandas numpy tqdm scipy lap pyyaml -q

import os, time, shutil, gc, yaml
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict, deque
from dataclasses import dataclass

import cv2
import numpy as np
import pandas as pd
import torch
import motmetrics as mm
from tqdm import tqdm
from ultralytics import YOLO
from google.colab import drive

try:
    torch.backends.cudnn.benchmark = True
except Exception:
    pass

drive.mount('/content/drive', force_remount=False)

DATASET_ROOT  = Path('/content/drive/MyDrive/visdrone/VisDrone_Zips/VisDrone2019-MOT-test-dev/VisDrone2019-MOT-test-dev')
SEQ_DIR       = DATASET_ROOT / 'sequences'
ANNOT_DIR     = DATASET_ROOT / 'annotations'
DRIVE_RESULTS = Path('/content/drive/MyDrive/concept experiment base 17 sequence')
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
print('All experiment outputs will be saved in:', DRIVE_RESULTS)
LOCAL_TMP     = Path('/content/_acmot_v10_tmp')

assert SEQ_DIR.exists() and ANNOT_DIR.exists(), 'Dataset path not found'
all_sequences = sorted([d for d in SEQ_DIR.iterdir() if d.is_dir()])

# ── Run only the five sequences missing from the preserved 12-sequence table ──
VAL_SEQS_NAMES = [
    'uav0000073_04464_v',
    'uav0000120_04775_v',
    'uav0000161_00000_v',
    'uav0000297_02761_v',
    'uav0000370_00001_v',
]
by_name  = {s.name: s for s in all_sequences}
VAL_SEQS = [by_name[n] for n in VAL_SEQS_NAMES if n in by_name]

MODEL_NAME = 'yolov8n.pt'
DEVICE     = '0' if torch.cuda.is_available() else 'cpu'
assert torch.cuda.is_available(), 'CUDA/T4 is required for the comparable legacy run'
assert 'T4' in torch.cuda.get_device_name(0), f'Tesla T4 required, found: {torch.cuda.get_device_name(0)}'
HALF       = True

print(f'AC-MOT v10 | Detector: {MODEL_NAME} | Device={DEVICE} | FP16={HALF}')
print(f'Running on {len(VAL_SEQS)} missing sequences:')
for s in VAL_SEQS:
    print(f'  - {s.name}')

In [ ]:
# ════════════════════════════════════════════════════════════════
#  CELL 2 — AC-MOT MODULES (v10 improved)
# ════════════════════════════════════════════════════════════════

@dataclass
class SceneState:
    sci: float = 0.0
    scene: str = 'clear'
    brightness: float = 128.0
    blur: float = 500.0
    edge_density: float = 0.0
    crowd: float = 0.0
    tiny_ratio: float = 0.0
    n_dets: int = 0


class SceneAnalyzer:
    """
    v10 fix: tightened crowd/SCI thresholds to stop over-classifying
    clear UAV sequences (brightness 120-200) as crowded.
    Key changes vs v9:
      - crowd > 0.65  (was 0.55)  — needs more objects before 'crowded'
      - edge_density > 0.13 (was 0.10) — less sensitive to texture
      - SCI crowd weight 0.35→0.30, edge weight 0.25→0.20,
        tiny weight 0.25→0.30 (tiny objects matter more for UAV)
    """
    def __init__(self, window: int = 7):
        self.sci_hist = deque(maxlen=window)

    def analyze(self, img: np.ndarray, prev_boxes: np.ndarray) -> SceneState:
        small = cv2.resize(img, (0, 0), fx=0.25, fy=0.25)
        gray  = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)

        brightness  = float(gray.mean())
        blur        = float(cv2.Laplacian(gray, cv2.CV_64F).var())
        edge_density= float(cv2.Canny(gray, 50, 120).mean() / 255.0)
        n_dets      = len(prev_boxes)
        crowd       = min(n_dets / 30.0, 1.0)          # v10: divisor 25→30

        if n_dets:
            areas      = ((prev_boxes[:, 2] - prev_boxes[:, 0]) *
                          (prev_boxes[:, 3] - prev_boxes[:, 1]))
            tiny_ratio = float(np.mean(areas < 32 * 32))
        else:
            tiny_ratio = 0.0

        # v10: rebalanced weights
        raw_sci = (0.30 * crowd
                 + 0.20 * min(edge_density / 0.14, 1.0)
                 + 0.30 * tiny_ratio)
        if brightness < 80:
            raw_sci += 0.10
        if blur < 180:
            raw_sci += 0.05

        self.sci_hist.append(float(np.clip(raw_sci, 0.0, 1.0)))
        sci = float(np.mean(self.sci_hist))

        # v10: tighter scene thresholds
        if brightness < 80:
            scene = 'night'
        elif blur < 180:
            scene = 'blur'
        elif tiny_ratio > 0.50:
            scene = 'tiny'
        elif crowd > 0.65 or edge_density > 0.13:   # v9 was 0.55 / 0.10
            scene = 'crowded'
        else:
            scene = 'clear'

        return SceneState(sci=sci, scene=scene, brightness=brightness,
                          blur=blur, edge_density=edge_density,
                          crowd=crowd, tiny_ratio=tiny_ratio, n_dets=n_dets)

    def reset(self):
        self.sci_hist.clear()


class SmartCalibrator:
    """
    v10 fix:
      - conf floor raised 0.17→0.19 (prevents FP explosion on UAV small objects)
      - conf ceiling kept 0.28 (safe for VisDrone)
      - imgsz thresholds unchanged (640/736/832 ladder)
    """
    def __init__(self, adaptive_threshold: bool = True,
                       adaptive_resolution: bool = True):
        self.adaptive_threshold  = adaptive_threshold
        self.adaptive_resolution = adaptive_resolution

    def params(self, state: SceneState) -> dict:
        conf  = 0.25
        iou   = 0.45
        imgsz = 640

        if self.adaptive_threshold:
            conf = 0.245 - 0.050 * state.sci          # v10: slope 0.055→0.050 (gentler)
            iou  = 0.490 - 0.050 * state.sci
            if state.scene in ['crowded', 'tiny', 'night']:
                conf -= 0.012                          # v10: nudge 0.015→0.012
            if state.scene == 'blur':
                iou -= 0.012

        if self.adaptive_resolution:
            if state.sci > 0.60 or state.tiny_ratio > 0.50:
                imgsz = 832
            elif state.sci > 0.35 or state.scene in ['crowded', 'tiny']:
                imgsz = 736

        return dict(
            conf  = float(np.clip(conf,  0.19, 0.28)),   # v10: floor 0.17→0.19
            iou   = float(np.clip(iou,   0.40, 0.52)),
            imgsz = int(imgsz)
        )


class LightweightReID:
    """
    Kept for ablation only — NOT used in production system in v10.
    v10 ablation proved ReID increases IDS on VisDrone (A3: 157 vs A2: 128).
    """
    def __init__(self, crop=24, bank=4, threshold=0.85, max_age=30):
        self.crop       = crop
        self.bank_size  = bank
        self.threshold  = threshold          # v10: raised 0.82→0.85 (stricter matching)
        self.max_age    = max_age            # v10: reduced 35→30 (shorter memory)
        self.bank       = defaultdict(lambda: deque(maxlen=bank))
        self.lost_feat  = {}
        self.lost_age   = {}
        self.seen_ids   = set()
        self.frame_idx  = 0

    def _feature(self, img, box):
        x1, y1, x2, y2 = [int(max(0, v)) for v in box]
        crop = img[y1:y2, x1:x2]
        if crop.size == 0:
            return None
        gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY) if crop.ndim == 3 else crop
        feat = cv2.resize(gray, (self.crop, self.crop)).ravel().astype(np.float32)
        feat -= feat.mean()
        norm = np.linalg.norm(feat)
        return feat / norm if norm > 1e-6 else None

    def update_and_remap(self, img, ids, boxes):
        self.frame_idx += 1
        current  = set(ids.tolist()) if len(ids) else set()
        remapped = ids.copy()

        for i, tid in enumerate(ids):
            tid  = int(tid)
            feat = self._feature(img, boxes[i])
            if feat is None:
                continue
            if tid not in self.seen_ids and self.lost_feat:
                best_tid, best_sim = tid, self.threshold
                for old_tid, old_feat in list(self.lost_feat.items()):
                    sim = float(np.dot(feat, old_feat))
                    if sim > best_sim:
                        best_tid, best_sim = old_tid, sim
                if best_tid != tid:
                    remapped[i] = best_tid
                    self.lost_feat.pop(best_tid, None)
                    self.lost_age.pop(best_tid, None)
                    tid = best_tid
            self.bank[tid].append(feat)
            self.seen_ids.add(tid)

        for tid in list(self.seen_ids):
            if tid not in current and tid not in self.lost_feat and self.bank[tid]:
                mean_feat = np.mean(np.stack(self.bank[tid]), axis=0)
                norm = np.linalg.norm(mean_feat)
                if norm > 1e-6:
                    self.lost_feat[tid] = mean_feat / norm
                    self.lost_age[tid]  = self.frame_idx

        for tid, age in list(self.lost_age.items()):
            if self.frame_idx - age > self.max_age:
                self.lost_feat.pop(tid, None)
                self.lost_age.pop(tid, None)
        return remapped


# ── Helper functions ────────────────────────────────────────────

def load_gt(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    cols = ['frame','id','x','y','w','h','score','cat','trunc','occ']
    df   = pd.read_csv(path, header=None, names=cols)
    df   = df[df['cat'].isin([1,4,5,6,9])]
    df   = df[(df['occ'] < 2) & (df['trunc'] < 2) & (df['score'] == 1)]
    return df.reset_index(drop=True)


def iou_dist(pred: np.ndarray, gt: np.ndarray) -> np.ndarray:
    if not len(pred) or not len(gt):
        return np.empty((len(gt), len(pred)))
    ix1   = np.maximum(pred[:, 0:1].T, gt[:, 0:1])
    iy1   = np.maximum(pred[:, 1:2].T, gt[:, 1:2])
    ix2   = np.minimum(pred[:, 2:3].T, gt[:, 2:3])
    iy2   = np.minimum(pred[:, 3:4].T, gt[:, 3:4])
    inter = np.maximum(0, ix2 - ix1) * np.maximum(0, iy2 - iy1)
    ap    = (pred[:, 2] - pred[:, 0]) * (pred[:, 3] - pred[:, 1])
    ag    = (gt[:, 2] - gt[:, 0]) * (gt[:, 3] - gt[:, 1])
    union = ap[np.newaxis, :] + ag[:, np.newaxis] - inter
    return 1.0 - np.where(union > 0, inter / union, 0.0)


def hota_approx(tp: int, fp: int, fn: int, ids: int) -> float:
    det_a = tp / max(tp + fp + fn, 1)
    ass_a = max(0.0, 1.0 - ids / max(tp, 1))
    return float(np.sqrt(det_a * ass_a))


def eval_acc(acc, name='seq') -> dict:
    mh   = mm.metrics.create()
    summ = mh.compute(acc,
        metrics=['mota','idf1','num_switches','recall','precision',
                 'num_misses','num_false_positives','num_matches'],
        name=name)
    row  = summ.iloc[0]
    return dict(
        mota      = float(row['mota']),
        idf1      = float(row['idf1']),
        recall    = float(row['recall']),
        precision = float(row['precision']),
        ids       = int(row['num_switches']),
        fn        = int(row['num_misses']),
        fp        = int(row['num_false_positives']),
        matches   = int(row['num_matches']),
        hota      = hota_approx(
            int(row['num_matches']), int(row['num_false_positives']),
            int(row['num_misses']),  int(row['num_switches'])),
    )


def build_tracker_yaml(name, high, low, new, buffer, match) -> str:
    path = Path(f'/content/{name}.yaml')
    data = dict(tracker_type='bytetrack',
                track_high_thresh=float(high),
                track_low_thresh=float(low),
                new_track_thresh=float(new),
                track_buffer=int(buffer),
                match_thresh=float(match),
                fuse_score=True)
    path.write_text(yaml.safe_dump(data, sort_keys=False), encoding='utf-8')
    return str(path)


def reset_tracker(model):
    if getattr(model, 'predictor', None) is not None:
        model.predictor = None


print('AC-MOT v10 modules ready')

In [ ]:
# ════════════════════════════════════════════════════════════════
#  CELL 3 — RUNNER + SYSTEMS DEFINITION
# ════════════════════════════════════════════════════════════════

TRACKERS = {
    # Official default ByteTrack — no tuning
    'baseline': 'bytetrack.yaml',
    # Tuned ByteTrack — used by both Baseline_TunedTracker AND AC-MOT_v10
    # high=0.18 : ByteTrack 2nd-round uses lower-conf detections → better recall
    # buffer=45 : longer track memory → fewer ID resets
    # match=0.86: stricter IoU matching → fewer wrong associations
    # new=0.20  : v10 fix, reduces spurious new tracks vs v9 (was 0.18)
    'acmot': build_tracker_yaml('bytetrack_v10_acmot',
                                high=0.18, low=0.04,
                                new=0.20,  buffer=45, match=0.86),
}

# ── 3 production systems ─────────────────────────────────────────
# System 1 vs System 2 → isolates: tracker YAML tuning contribution
# System 2 vs System 3 → isolates: adaptive scene intelligence contribution
SYSTEMS = [
    dict(name='Baseline_Default',
         model=MODEL_NAME, tracker='baseline',
         adaptive_threshold=False, adaptive_resolution=False,
         scene_analysis=False, reid=False),

    dict(name='Baseline_TunedTracker',
         model=MODEL_NAME, tracker='acmot',       # same tuned yaml as AC-MOT
         adaptive_threshold=False, adaptive_resolution=False,
         scene_analysis=False, reid=False),        # zero adaptive logic

    dict(name='AC-MOT_v10',
         model=MODEL_NAME, tracker='acmot',
         adaptive_threshold=True, adaptive_resolution=True,
         scene_analysis=True, reid=False),         # ReID OFF — v9 ablation proved it increases IDS
]

# ── Ablation systems (4 sequences) ──────────────────────────────
# A0 → A1: What does tuned YAML alone give?
# A1 → A2: What does adaptive threshold add on top of tuned YAML?
# A2 → A3: What does adaptive resolution add?
ABLATION_SYSTEMS = [
    dict(name='A0_Baseline_Default',
         model=MODEL_NAME, tracker='baseline',
         adaptive_threshold=False, adaptive_resolution=False,
         scene_analysis=False, reid=False),

    dict(name='A1_TunedTracker',
         model=MODEL_NAME, tracker='acmot',
         adaptive_threshold=False, adaptive_resolution=False,
         scene_analysis=False, reid=False),

    dict(name='A2_AdaptThreshold',
         model=MODEL_NAME, tracker='acmot',
         adaptive_threshold=True,  adaptive_resolution=False,
         scene_analysis=True,  reid=False),

    dict(name='A3_AdaptResolution',   # = production config
         model=MODEL_NAME, tracker='acmot',
         adaptive_threshold=True,  adaptive_resolution=True,
         scene_analysis=True,  reid=False),

]


def get_params(system: dict, state: SceneState) -> dict:
    if system.get('adaptive_threshold') or system.get('adaptive_resolution'):
        return SmartCalibrator(
            adaptive_threshold  = system.get('adaptive_threshold', False),
            adaptive_resolution = system.get('adaptive_resolution', False)
        ).params(state)
    return dict(conf=0.25, iou=0.45, imgsz=640)


def run_system(system: dict, seqs, run_tag: str) -> pd.DataFrame:
    model = YOLO(system['model'])
    if HALF:
        model.model.half()

    rows = []
    for seq in tqdm(seqs, desc=system['name']):
        gt         = load_gt(ANNOT_DIR / f'{seq.name}.txt')
        frames_drv = sorted(seq.glob('*.jpg'))
        if gt.empty or not frames_drv:
            continue

        LOCAL_TMP.mkdir(exist_ok=True)
        local_seq = LOCAL_TMP / seq.name
        if local_seq.exists():
            shutil.rmtree(local_seq)
        shutil.copytree(seq, local_seq)
        frames = sorted(local_seq.glob('*.jpg'))

        reset_tracker(model)
        calibrator   = SmartCalibrator(
                            system.get('adaptive_threshold', False),
                            system.get('adaptive_resolution', False))
        analyzer     = SceneAnalyzer()
        reid         = LightweightReID() if system.get('reid', False) else None
        acc          = mm.MOTAccumulator(auto_id=True)
        times        = []
        prev_boxes   = np.empty((0, 4))
        state        = SceneState()
        scene_counts = Counter()
        imgsz_log, conf_log = [], []

        for idx, fp in enumerate(frames, start=1):
            t0  = time.perf_counter()
            img = cv2.imread(str(fp))
            if img is None:
                continue

            if system.get('scene_analysis') and (idx == 1 or idx % 10 == 1):
                state = analyzer.analyze(img, prev_boxes)

            scene_counts[state.scene] += 1
            params = calibrator.params(state)
            imgsz_log.append(params['imgsz'])
            conf_log.append(params['conf'])

            res = model.track(
                source  = img,
                tracker = TRACKERS[system['tracker']],
                conf    = params['conf'],
                iou     = params['iou'],
                imgsz   = params['imgsz'],
                half    = HALF,
                persist = True,
                verbose = False,
                device  = DEVICE,
            )
            times.append(time.perf_counter() - t0)

            if res[0].boxes.id is not None:
                pred_ids   = res[0].boxes.id.cpu().numpy().astype(int)
                pred_boxes = res[0].boxes.xyxy.cpu().numpy()
            else:
                pred_ids   = np.array([], dtype=int)
                pred_boxes = np.empty((0, 4))
            prev_boxes = pred_boxes.copy()

            if reid is not None and len(pred_ids):
                pred_ids = reid.update_and_remap(img, pred_ids, pred_boxes)

            gt_f     = gt[gt['frame'] == idx]
            gt_ids   = gt_f['id'].values
            gt_boxes = (np.column_stack([
                gt_f['x'].values, gt_f['y'].values,
                gt_f['x'].values + gt_f['w'].values,
                gt_f['y'].values + gt_f['h'].values,
            ]) if len(gt_f) else np.empty((0, 4)))

            dist = iou_dist(pred_boxes, gt_boxes)
            acc.update(gt_ids, pred_ids,
                       dist if dist.size else np.empty((len(gt_ids), len(pred_ids))))

        shutil.rmtree(local_seq, ignore_errors=True)
        metrics = eval_acc(acc, seq.name)
        fps     = 1.0 / np.mean(times) if times else 0.0
        dom     = scene_counts.most_common(1)[0][0] if scene_counts else 'unknown'

        rows.append(dict(
            run_tag        = run_tag,
            system         = system['name'],
            tracker_cfg    = system['tracker'],       # logged for reproducibility
            sequence       = seq.name,
            frames         = len(frames),
            fps            = round(fps, 2),
            dominant_scene = dom,
            mean_imgsz     = round(float(np.mean(imgsz_log)), 1) if imgsz_log else 640,
            mean_conf      = round(float(np.mean(conf_log)),  4) if conf_log  else 0.25,
            **metrics,
        ))
        tqdm.write(
            f"{system['name']:<34} {seq.name[:24]:24s} "
            f"MOTA={metrics['mota']:.3f} IDF1={metrics['idf1']:.3f} "
            f"HOTA={metrics['hota']:.3f} IDS={metrics['ids']:4d} "
            f"FPS={fps:.1f} scene={dom}"
        )

    if HALF:
        torch.cuda.empty_cache()
    gc.collect()
    return pd.DataFrame(rows)


def summarize(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for system, g in df.groupby('system', sort=False):
        rows.append(dict(
            system     = system,
            sequences  = len(g),
            mota       = g['mota'].mean(),
            idf1       = g['idf1'].mean(),
            recall     = g['recall'].mean(),
            precision  = g['precision'].mean(),
            hota       = g['hota'].mean(),
            ids        = int(g['ids'].sum()),
            fn         = int(g['fn'].sum()),
            fp         = int(g['fp'].sum()),
            matches    = int(g['matches'].sum()),
            fps        = g['fps'].mean(),
            mean_imgsz = g['mean_imgsz'].mean(),
        ))
    out = pd.DataFrame(rows)
    if len(out) >= 2:
        base = out.iloc[0]
        for col in ['mota','idf1','recall','precision','hota','fps']:
            out[col + '_delta'] = out[col] - float(base[col])
        out['ids_delta'] = out['ids'] - int(base['ids'])
        out['fn_delta']  = out['fn']  - int(base['fn'])
        out['fp_delta']  = out['fp']  - int(base['fp'])
    return out


print('Runner ready')
print('Systems: Baseline_Default | Baseline_TunedTracker | AC-MOT_v10')
print('Tracker configs logged per-row for reproducibility')

In [ ]:
# CELL 4 — RUN THE FOUR-WAY ABLATION ON THE FIVE MISSING SEQUENCES

from datetime import datetime

run_tag = f'acmot_v10_ablation_missing5_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
new_rows = []
for system in ABLATION_SYSTEMS:
    new_rows.append(run_system(system, VAL_SEQS, run_tag))

new_per_sequence = pd.concat(new_rows, ignore_index=True)
assert set(new_per_sequence['sequence']) == set(VAL_SEQS_NAMES)
assert set(new_per_sequence['system']) == {
    'A0_Baseline_Default', 'A1_TunedTracker',
    'A2_AdaptThreshold', 'A3_AdaptResolution',
}
new_path = DRIVE_RESULTS / f'{run_tag}_per_sequence.csv'
new_per_sequence.to_csv(new_path, index=False)
print('Saved new five-sequence rows ->', new_path)


In [ ]:
# CELL 5 — FIND THE OLD 12-SEQUENCE PER-SEQUENCE FILES

# Put these two preserved CSVs anywhere under MyDrive, or set the paths manually.
OLD_NAMES = [
    'FINAL_RUN1_A0A1_20260603_081343_per_seq.csv',
    'FINAL_RUN2_A2A3_20260603_080537_per_seq.csv',
]
OLD_12_PER_SEQUENCE_CSVS = []
for name in OLD_NAMES:
    matches = list(Path('/content/drive/MyDrive').rglob(name))
    if len(matches) != 1:
        raise FileNotFoundError(
            f'Expected exactly one old CSV named {name}, found {len(matches)}. '
            'Upload/copy the preserved 12-sequence per-sequence CSV to MyDrive, then rerun this cell.'
        )
    OLD_12_PER_SEQUENCE_CSVS.append(matches[0])
print('Using old inputs:')
for p in OLD_12_PER_SEQUENCE_CSVS:
    print(' -', p)


In [ ]:
# CELL 6 — VALIDATE AND MERGE TO 17 SEQUENCES

old_frames = [pd.read_csv(p) for p in OLD_12_PER_SEQUENCE_CSVS]
old = pd.concat(old_frames, ignore_index=True)
name_map = {
    'A1_TunedTracker_Only': 'A1_TunedTracker',
    'A2_TunedTracker_AdaptThresh': 'A2_AdaptThreshold',
    'A3_TunedTracker_AdaptThresh_Res': 'A3_AdaptResolution',
    'Baseline_Default': 'A0_Baseline_Default',
    'Baseline_TunedTracker': 'A1_TunedTracker',
    'AC-MOT_v10': 'A3_AdaptResolution',
}
old['system'] = old['system'].replace(name_map)
required = {'A0_Baseline_Default','A1_TunedTracker','A2_AdaptThreshold','A3_AdaptResolution'}
old12 = old[old['system'].isin(required)].copy()
assert old12['sequence'].nunique() == 12, sorted(old12['sequence'].unique())
assert not set(old12['sequence']).intersection(VAL_SEQS_NAMES)
assert set(old12['system']) == required
assert not set(new_per_sequence['sequence']).intersection(set(old12['sequence']))
merged = pd.concat([old12, new_per_sequence], ignore_index=True)
assert merged['sequence'].nunique() == 17
assert not merged.duplicated(['system','sequence']).any()
assert set(merged['system']) == required

summary = []
for system, g in merged.groupby('system', sort=False):
    summary.append(dict(
        system=system, sequences=int(g['sequence'].nunique()),
        mota=float(g['mota'].mean()), idf1=float(g['idf1'].mean()),
        hota=float(g['hota'].mean()), recall=float(g['recall'].mean()),
        precision=float(g['precision'].mean()), ids=int(g['ids'].sum()),
        fn=int(g['fn'].sum()), fp=int(g['fp'].sum()), fps=float(g['fps'].mean()),
    ))
summary = pd.DataFrame(summary)
base = summary.iloc[0]
summary['mota_delta'] = summary['mota'] - float(base['mota'])
summary['idf1_delta'] = summary['idf1'] - float(base['idf1'])
summary['hota_delta'] = summary['hota'] - float(base['hota'])
summary['recall_delta'] = summary['recall'] - float(base['recall'])
summary['precision_delta'] = summary['precision'] - float(base['precision'])
summary['fps_delta'] = summary['fps'] - float(base['fps'])
summary['ids_delta'] = summary['ids'] - int(base['ids'])
summary['ids_reduction'] = int(base['ids']) - summary['ids']
summary['ids_reduction_pct'] = (int(base['ids']) - summary['ids']) / max(1, int(base['ids'])) * 100.0
summary['realtime_20fps'] = summary['fps'] >= 20.0
summary['strict_realtime_25fps'] = summary['fps'] >= 25.0

# Report separate winners; do not hide trade-offs between FPS, quality and IDS.
realtime = summary[summary['realtime_20fps']].copy()
best_realtime = (realtime.sort_values(['mota','hota','ids','fps'], ascending=[False,False,True,False]).iloc[0]['system'] if len(realtime) else None)
best_mota = summary.loc[summary['mota'].idxmax(), 'system']
best_hota = summary.loc[summary['hota'].idxmax(), 'system']
best_ids = summary.loc[summary['ids'].idxmin(), 'system']
best_fps = summary.loc[summary['fps'].idxmax(), 'system']

def dominates(a, b):
    return (a['mota'] >= b['mota'] and a['hota'] >= b['hota'] and a['ids'] <= b['ids'] and a['fps'] >= b['fps'] and
            (a['mota'] > b['mota'] or a['hota'] > b['hota'] or a['ids'] < b['ids'] or a['fps'] > b['fps']))

pareto = [a['system'] for _, a in summary.iterrows()
          if not any(dominates(b, a) for _, b in summary.iterrows() if b['system'] != a['system'])]
acmot = summary[summary['system'] == 'A3_AdaptResolution'].iloc[0]
acmot_dominates_all = all(dominates(acmot, b) for _, b in summary.iterrows() if b['system'] != 'A3_AdaptResolution')

stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
merged_path = DRIVE_RESULTS / f'acmot_v10_ablation_merged17_{stamp}_per_sequence.csv'
summary_path = DRIVE_RESULTS / f'acmot_v10_ablation_merged17_{stamp}_summary.csv'
manifest_path = DRIVE_RESULTS / f'acmot_v10_ablation_merged17_{stamp}_manifest.json'
merged.to_csv(merged_path, index=False)
summary.to_csv(summary_path, index=False)
manifest_path.write_text(json.dumps({
    'protocol': 'AC-MOT legacy v10 four-way ablation',
    'old_sequence_count': 12, 'new_sequence_count': 5, 'merged_sequence_count': 17,
    'new_sequences': VAL_SEQS_NAMES,
    'old_input_csvs': [str(p) for p in OLD_12_PER_SEQUENCE_CSVS],
    'new_input_csv': str(new_path),
    'duplicate_check': 'passed',
    'metric_note': 'macro means for rates; sums for IDS/FN/FP; old inputs untouched',
    'realtime_policy': 'acceptable realtime is >=20 FPS; strict realtime is >=25 FPS',
    'best_realtime_20fps': best_realtime,
    'best_mota': best_mota,
    'best_hota': best_hota,
    'lowest_ids': best_ids,
    'best_fps': best_fps,
    'pareto_frontier': pareto,
    'acmot_dominates_all_objectives': bool(acmot_dominates_all),
}, indent=2) + '\n')
print('\nMERGED 17-SEQUENCE ABLATION')
print(summary.to_string(index=False, float_format=lambda x: f'{x:.4f}'))
print('\nOBJECTIVE WINNERS')
print('Best acceptable realtime (>=20 FPS):', best_realtime or 'NONE')
print('Best MOTA:', best_mota)
print('Best HOTA:', best_hota)
print('Lowest IDS:', best_ids)
print('Best FPS:', best_fps)
print('Pareto frontier:', ', '.join(pareto))
print('AC-MOT dominates all objectives:', acmot_dominates_all)
print('\nSaved:', merged_path, summary_path, manifest_path, sep='\n')


In [ ]:
# CELL 7 — EXPORT PARAMETERS, EXCEL COMPARISON AND CHARTS
# Run this after CELL 6. It formats only the real merged17 results.
!pip -q install XlsxWriter
import matplotlib.pyplot as plt

excel_path = DRIVE_RESULTS / f'acmot_v10_ablation_merged17_{stamp}_comparison.xlsx'
chart_path = DRIVE_RESULTS / f'acmot_v10_ablation_merged17_{stamp}_comparison.png'
parameter_rows = [
    dict(system='A0_Baseline_Default', tracker='baseline', adaptive_threshold=False, adaptive_resolution=False, scene_analysis=False, reid=False, protocol_note='Default baseline; fixed settings'),
    dict(system='A1_TunedTracker', tracker='acmot', adaptive_threshold=False, adaptive_resolution=False, scene_analysis=False, reid=False, protocol_note='Tuned tracker only'),
    dict(system='A2_AdaptThreshold', tracker='acmot', adaptive_threshold=True, adaptive_resolution=False, scene_analysis=True, reid=False, protocol_note='Tuned tracker + adaptive threshold'),
    dict(system='A3_AdaptResolution', tracker='acmot', adaptive_threshold=True, adaptive_resolution=True, scene_analysis=True, reid=False, protocol_note='Full AC-MOT adaptive threshold + resolution'),
]
parameters = pd.DataFrame(parameter_rows)
final_table = summary.copy()
metric_cols = ['mota','idf1','hota','recall','precision','fps','ids']
step_effect = final_table[['system'] + metric_cols].copy()
step_effect.insert(1, 'previous_system', ['—'] + final_table['system'].iloc[:-1].tolist())
for metric in metric_cols:
    step_effect[f'{metric}_change_vs_previous'] = final_table[metric].diff()
strict = summary[summary['strict_realtime_25fps']]
best_strict = (strict.sort_values(['mota','hota','ids','fps'], ascending=[False,False,True,False]).iloc[0]['system'] if len(strict) else 'NONE')
winners = pd.DataFrame([
    {'objective':'Best acceptable realtime (>=20 FPS)','winner':best_realtime or 'NONE'},
    {'objective':'Best strict realtime (>=25 FPS)','winner':best_strict},
    {'objective':'Best MOTA','winner':best_mota}, {'objective':'Best HOTA','winner':best_hota},
    {'objective':'Lowest IDS','winner':best_ids}, {'objective':'Best FPS','winner':best_fps},
    {'objective':'Pareto frontier','winner':', '.join(pareto)},
    {'objective':'AC-MOT dominates all objectives','winner':bool(acmot_dominates_all)},
])
with pd.ExcelWriter(excel_path, engine='xlsxwriter') as writer:
    final_table.to_excel(writer, sheet_name='Summary', index=False)
    merged.to_excel(writer, sheet_name='Per_Sequence', index=False)
    parameters.to_excel(writer, sheet_name='Parameters', index=False)
    step_effect.to_excel(writer, sheet_name='Step_Effects', index=False)
    winners.to_excel(writer, sheet_name='Objective_Winners', index=False)
    manifest_df = pd.DataFrame([{'merged_csv':str(merged_path),'summary_csv':str(summary_path),'manifest_json':str(manifest_path),'sequence_count':int(merged['sequence'].nunique()),'metric_protocol':'macro mean rates; sum IDS/FN/FP'}])
    manifest_df.to_excel(writer, sheet_name='Manifest', index=False)
    wb = writer.book; header = wb.add_format({'bold':True,'bg_color':'#1F4E78','font_color':'white'})
    for name, frame in [('Summary',final_table),('Per_Sequence',merged),('Parameters',parameters),('Step_Effects',step_effect),('Objective_Winners',winners),('Manifest',manifest_df)]:
        ws = writer.sheets[name]; ws.freeze_panes(1,0); ws.autofilter(0,0,len(frame),max(0,len(frame.columns)-1)); ws.set_column(0,max(0,len(frame.columns)-1),18)
        for j, col in enumerate(frame.columns): ws.write(0,j,col,header)
    charts = wb.add_worksheet('Charts'); charts.write('A1','17-sequence AC-MOT ablation comparison'); charts.write('A2','Generated from the actual merged summary of this run.')
    charts.insert_chart('A4', {'type':'column','title':{'name':'MOTA / HOTA'},'categories':"='Summary'!$A$2:$A$5",'series':[{'name':'MOTA','values':"='Summary'!$C$2:$C$5"},{'name':'HOTA','values':"='Summary'!$E$2:$E$5"}]})
    charts.insert_chart('J4', {'type':'column','title':{'name':'FPS / IDS'},'categories':"='Summary'!$A$2:$A$5",'series':[{'name':'FPS','values':"='Summary'!$K$2:$K$5"},{'name':'IDS','values':"='Summary'!$H$2:$H$5"}]})
fig, axes = plt.subplots(1,4,figsize=(18,4))
for ax, metric, title in zip(axes,['mota','hota','ids','fps'],['MOTA','HOTA','IDS','FPS']):
    ax.bar(summary['system'],summary[metric],color=['#777777','#4C78A8','#F58518','#54A24B']); ax.set_title(title); ax.tick_params(axis='x',rotation=35); ax.grid(axis='y',alpha=.25)
fig.tight_layout(); fig.savefig(chart_path,dpi=180,bbox_inches='tight'); plt.close(fig)
print('Saved Excel:', excel_path)
print('Saved chart:', chart_path)
